## Ana Paula

In [2]:
import pandas as pd
df = pd.read_csv('data/fake_job_postings.csv')
print(df.shape)
print(df.head())

(17880, 18)
   job_id                                      title            location  \
0       1                           Marketing Intern    US, NY, New York   
1       2  Customer Service - Cloud Video Production      NZ, , Auckland   
2       3    Commissioning Machinery Assistant (CMA)       US, IA, Wever   
3       4          Account Executive - Washington DC  US, DC, Washington   
4       5                        Bill Review Manager  US, FL, Fort Worth   

  department salary_range                                    company_profile  \
0  Marketing          NaN  We're Food52, and we've created a groundbreaki...   
1    Success          NaN  90 Seconds, the worlds Cloud Video Production ...   
2        NaN          NaN  Valor Services provides Workforce Solutions th...   
3      Sales          NaN  Our passion for improving quality of life thro...   
4        NaN          NaN  SpotSource Solutions LLC is a Global Human Cap...   

                                         descripti

In [3]:
# Eliminar columnas que no aportan información al modelo
columnas_a_eliminar = ['job_id'] # El ID es solo un contador
df = df.drop(columns=columnas_a_eliminar)

In [4]:
# Rellenar nulos en columnas categóricas y de texto
categoricas_y_texto = ['department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'employment_type', 'required_experience', 'required_education', 'industry', 'function']

for col in categoricas_y_texto:
    df[col] = df[col].fillna('Unspecified')

In [5]:
import re

def limpiar_texto(texto):
    if pd.isna(texto) or texto == 'Unspecified':
        return ''
    texto = str(texto).lower() # Todo a minúsculas
    texto = re.sub(r'[^a-z0-9\s]', '', texto) # Quitar caracteres especiales (deja solo letras, números y espacios)
    texto = re.sub(r'\s+', ' ', texto).strip() # Limpiar espacios dobles o saltos de línea
    return texto

# Ejemplo: Aplicar la limpieza a la columna 'description'
df['description_clean'] = df['description'].apply(limpiar_texto)

In [6]:
# Extraer el código del país (los dos primeros caracteres antes de la primera coma)
df['country'] = df['location'].apply(lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unspecified')

In [7]:
# Ver la distribución de clases
print(df['fraudulent'].value_counts())
print(df['fraudulent'].value_counts(normalize=True) * 100)

fraudulent
0    17014
1      866
Name: count, dtype: int64
fraudulent
0    95.1566
1     4.8434
Name: proportion, dtype: float64


In [10]:
# 1. Comprobar que ya no quedan nulos en las columnas que vas a usar
print(df[['title', 'description_clean', 'country', 'fraudulent']].isnull().sum())

# 2. Ver cómo ha quedado una fila real tras la limpieza
print(df['description_clean'].iloc[0][:300]) # Muestra los primeros 300 caracteres de la primera descripción

title                0
description_clean    0
country              0
fraudulent           0
dtype: int64
food52 a fastgrowing james beard awardwinning online food community and crowdsourced and curated recipe hub is currently interviewing full and parttime unpaid interns to work in a small team of editors executives and developers in its new york city headquartersreproducing andor repackaging existing 


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Configurar el vectorizador para extraer las 500 palabras clave limpias
tfidf = TfidfVectorizer(max_features=500, stop_words='english')

# 2. Aplicar el vectorizador a la columna de texto y guardarlo en la variable X_tfidf
X_tfidf = tfidf.fit_transform(df['description_clean']).toarray()

# 3. Comprobar que se ha creado la matriz correctamente
print("Forma de la matriz de texto:", X_tfidf.shape)

Forma de la matriz de texto: (17880, 500)


In [ ]:
# 1. Convertir la matriz TF-IDF en un DataFrame de Pandas
df_tfidf = pd.DataFrame(X_tfidf, columns=[f"txt_{word}" for word in tfidf.get_feature_names_out()])

# Nota: tiene txt al inicio para saber que se saco de la descripcion

# 2. Seleccionar las otras columnas numéricas/binarias que interesan del df original
columnas_numericas = ['telecommuting', 'has_company_logo', 'has_questions']
df_numerico = df[columnas_numericas].reset_index(drop=True)

# 3. Concatenar (juntar de lado) las columnas numéricas y las del texto
X_final = pd.concat([df_numerico, df_tfidf], axis=1)

# 4. Tu variable objetivo (lo que queremos predecir) sigue siendo la misma
y_final = df['fraudulent'].reset_index(drop=True)

# 5. Comprobar las dimensiones finales
print("Dimensiones de las variables predictoras (X):", X_final.shape)
print("Dimensiones de la variable objetivo (y):", y_final.shape)

Dimensiones de las variables predictoras (X): (17880, 503)
Dimensiones de la variable objetivo (y): (17880,)


In [18]:
# Ver las primeras 50 palabras que corresponden a esas 500 columnas
print(tfidf.get_feature_names_out()[:50])

['200' 'ability' 'able' 'account' 'accounting' 'accounts' 'accurate'
 'achieve' 'activities' 'administrative' 'advertising' 'agency' 'agile'
 'amp' 'analysis' 'analytics' 'andor' 'app' 'application' 'applications'
 'apply' 'approach' 'appropriate' 'architecture' 'area' 'areas' 'asia1500'
 'aspects' 'assigned' 'assist' 'assistant' 'attention' 'available'
 'background' 'base' 'based' 'basis' 'believe' 'benefits' 'best' 'better'
 'big' 'brand' 'brands' 'bring' 'build' 'building' 'business' 'businesses'
 'calls']
